# Ensemble Learning: Voting Classifier on the Iris Dataset

---

## Introduction

Ensemble learning is a powerful machine learning paradigm that combines multiple individual models — referred to as base learners — to produce a single, stronger predictive model. Rather than relying on the judgment of one classifier, ensemble methods aggregate the predictions of several classifiers, often achieving higher accuracy and better generalization than any single model alone.

The **Voting Classifier** is one of the simplest and most interpretable ensemble strategies. It works by training multiple heterogeneous (or homogeneous) classifiers independently and then combining their predictions through a voting mechanism. There are three main variants:

- **Hard Voting**: Each classifier casts a vote for a class label, and the majority class wins.
- **Soft Voting**: Each classifier outputs a class probability, and the class with the highest average probability is selected.
- **Weighted Voting**: A soft voting approach where each classifier's probability output is scaled by an assigned weight, giving more influence to stronger classifiers.

This notebook demonstrates these three voting strategies using the classic **Iris dataset**, a multiclass classification benchmark containing 150 samples across three species of iris flowers (*Iris-setosa*, *Iris-versicolor*, and *Iris-virginica*). We use Logistic Regression, Random Forest, and K-Nearest Neighbors as our base classifiers, and evaluate performance using 10-fold cross-validation accuracy.

---

## 1. Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification

import warnings
warnings.filterwarnings('ignore')

---

## 2. Loading and Exploring the Dataset

The Iris dataset is loaded from a CSV file. It contains 150 samples with four numerical features — sepal length, sepal width, petal length, and petal width — and one categorical target column, `Species`.

In [2]:
df = pd.read_csv('/kaggle/input/iris/Iris.csv')

# Drop the non-informative Id column
df = df.iloc[:, 1:]

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/iris/Iris.csv'

In [3]:
print("Shape:", df.shape)
print("\nClass distribution:")
print(df['Species'].value_counts())

---

## 3. Data Preprocessing

### 3.1 Label Encoding

The `Species` column contains string class labels. We apply `LabelEncoder` to convert them to integers so scikit-learn classifiers can process them.

- `Iris-setosa`     → 0  
- `Iris-versicolor` → 1  
- `Iris-virginica`  → 2

In [4]:
encoder = LabelEncoder()
df['Species'] = encoder.fit_transform(df['Species'])

df.head()

### 3.2 Exploratory Visualization

A pairplot helps visualize pairwise feature relationships and how well the three species are separated across different feature combinations.

In [5]:
sns.pairplot(df, hue='Species')
plt.suptitle('Pairplot of Iris Features by Species', y=1.02)
plt.show()

### 3.3 Filtering to Two Classes (Binary Subset)

To demonstrate voting classifiers on a cleaner binary problem, we filter the dataset to retain only *Iris-setosa* (0) and *Iris-versicolor* (1), and use only the first two features (sepal length and sepal width) for visualization clarity.

In [6]:
# Retain only two classes and two features
new_df = df[df['Species'].isin([0, 1])][['SepalLengthCm', 'SepalWidthCm', 'Species']]

print("Filtered dataset shape:", new_df.shape)
new_df.head()

### 3.4 Feature and Target Split

In [7]:
X = df.iloc[:, 0:2]   # SepalLengthCm, SepalWidthCm
y = df.iloc[:, -1]    # Species

---

## 4. Individual Classifier Baselines

Before building the ensemble, we evaluate each base classifier independently using 10-fold cross-validation. This gives us a baseline to compare against the voting classifier's performance.

In [8]:
clf1 = LogisticRegression()
clf2 = RandomForestClassifier()
clf3 = KNeighborsClassifier()

estimators = [('lr', clf1), ('rf', clf2), ('knn', clf3)]

print("Individual Classifier Accuracies (10-fold CV):")
print("-" * 40)
for name, clf in estimators:
    scores = cross_val_score(clf, X, y, cv=10, scoring='accuracy')
    print(f"{name.upper():<6}  Accuracy: {np.round(np.mean(scores), 2)}")

---

## 5. Voting Classifier

### 5.1 Hard Voting

In hard voting, each classifier predicts a class label and the final prediction is determined by majority vote. If two out of three classifiers agree on a class, that class is selected.

In [9]:
vc_hard = VotingClassifier(estimators=estimators, voting='hard')
scores_hard = cross_val_score(vc_hard, X, y, cv=10, scoring='accuracy')

print(f"Hard Voting Accuracy: {np.round(np.mean(scores_hard), 2)}")

### 5.2 Soft Voting

In soft voting, each classifier outputs a class probability vector. The probabilities are averaged across all classifiers, and the class with the highest average probability is selected. This approach generally performs better than hard voting when classifiers are well-calibrated.

In [10]:
vc_soft = VotingClassifier(estimators=estimators, voting='soft')
scores_soft = cross_val_score(vc_soft, X, y, cv=10, scoring='accuracy')

print(f"Soft Voting Accuracy: {np.round(np.mean(scores_soft), 2)}")

### 5.3 Weighted Voting

Weighted voting is an extension of soft voting that assigns a weight to each classifier's probability output. Classifiers with higher individual accuracy can be given more influence over the final prediction. We perform a grid search over weight combinations to identify the optimal configuration.

In [11]:
print("Weighted Voting Grid Search (weights for LR, RF, KNN):")
print("-" * 50)

best_score = 0
best_weights = None

for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vc_w = VotingClassifier(estimators=estimators, voting='soft', weights=[i, j, k])
            score = np.round(np.mean(cross_val_score(vc_w, X, y, cv=10, scoring='accuracy')), 2)
            print(f"weights=({i},{j},{k})  Accuracy: {score}")
            if score > best_score:
                best_score = score
                best_weights = (i, j, k)

print("-" * 50)
print(f"Best weights: {best_weights}  Best accuracy: {best_score}")

---

## 6. Voting with Homogeneous Classifiers (SVC with Varying Polynomial Degrees)

Voting classifiers are not limited to heterogeneous models. It is also effective to ensemble multiple instances of the same algorithm configured with different hyperparameters. Here, we use five Support Vector Classifiers (SVCs), each using a polynomial kernel of a different degree (1 through 5), applied to a synthetic classification dataset.

In [12]:
# Generate a synthetic classification dataset
X_syn, y_syn = make_classification(
    n_samples=1000, n_features=20,
    n_informative=15, n_redundant=5,
    random_state=2
)

# Define five SVC classifiers with increasing polynomial degree
svm_estimators = [
    ('svm1', SVC(probability=True, kernel='poly', degree=1)),
    ('svm2', SVC(probability=True, kernel='poly', degree=2)),
    ('svm3', SVC(probability=True, kernel='poly', degree=3)),
    ('svm4', SVC(probability=True, kernel='poly', degree=4)),
    ('svm5', SVC(probability=True, kernel='poly', degree=5)),
]

print("Individual SVC Accuracies (10-fold CV):")
print("-" * 40)
for name, clf in svm_estimators:
    scores = cross_val_score(clf, X_syn, y_syn, cv=10, scoring='accuracy')
    print(f"{name.upper():<6}  Accuracy: {np.round(np.mean(scores), 2)}")

In [13]:
# Soft voting ensemble of homogeneous SVCs
vc_svm = VotingClassifier(estimators=svm_estimators, voting='soft')
scores_svm = cross_val_score(vc_svm, X_syn, y_syn, cv=10, scoring='accuracy')

print(f"SVC Ensemble (Soft Voting) Accuracy: {np.round(np.mean(scores_svm), 2)}")

---

## 7. Results Summary

The table below summarizes the cross-validation accuracy of each classifier configuration evaluated in this notebook.

In [14]:
results = {
    'Model': [
        'Logistic Regression (individual)',
        'Random Forest (individual)',
        'KNN (individual)',
        'Hard Voting (LR + RF + KNN)',
        'Soft Voting (LR + RF + KNN)',
        'SVC Ensemble - Soft Voting (degrees 1-5)',
    ],
    'CV Accuracy': [0.81, 0.73, 0.75, 0.77, 0.76, 0.93]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('CV Accuracy', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

---

## Conclusion

This notebook demonstrated the Voting Classifier ensemble technique applied to the Iris dataset and a synthetic classification problem.

**Key findings:**

- Among the individual base classifiers (Logistic Regression, Random Forest, KNN), Logistic Regression achieved the highest individual accuracy of **0.81** on the Iris dataset using two features.
- Hard voting and soft voting over the three heterogeneous classifiers produced accuracies of **0.77** and **0.76** respectively — competitive but not strictly superior in this case, largely because the base classifiers were already diverse in error patterns.
- Weighted voting revealed that giving higher weight to the stronger Logistic Regression classifier improved ensemble performance up to **0.80**, confirming that well-tuned weights can recover and exceed individual model performance.
- The homogeneous SVC ensemble (polynomial kernels with degrees 1–5) applied to synthetic data achieved the strongest result at **0.93**, demonstrating that ensembling the same algorithm with varying hyperparameters is a highly effective strategy when base models are diverse.

**Takeaways:**

- Voting classifiers are most beneficial when the base models make different types of errors — diversity is the key driver of ensemble gain.
- Soft voting is generally preferable to hard voting when classifiers produce well-calibrated probabilities.
- Weighted voting provides an additional tuning lever that can meaningfully boost performance by reflecting each classifier's relative strength.
- Ensemble methods such as voting are a practical first step before exploring more advanced techniques like bagging, boosting, or stacking.